# Jaguar Re-Identification Baseline


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

# Add repo root to path so we can import src
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
from src.utils import get_device, set_seed
from src.models import ArcFaceModel
from src.training import fit
from src.wandb_utils import init_wandb, log_checkpoint_artifact, log_submission_artifact

# Load environment variables from .env file
load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
print(f"Device: {device}")


In [ ]:
!wandb login


## Configuration


In [ ]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("checkpoints"),

    # Model
    "backbone_name": "hf-hub:BVRA/MegaDescriptor-L-384",
    "input_size": 384,
    "embedding_dim": 256,
    "hidden_dim": 512,
    "freeze_backbone": True,

    # ArcFace
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,

    # Training
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 50,
    "patience": 10,
    "val_split": 0.2,
    "num_workers": 2,

    # Reproducibility
    "seed": RANDOM_SEED,

    # Normalization
    "mean": data.DEFAULT_MEAN,
    "std": data.DEFAULT_STD,
}

config["checkpoint_dir"].mkdir(exist_ok=True)


## Load Data

In [ ]:

train_df = data.load_train_df(config["data_dir"])
train_df, label_encoder = data.encode_labels(train_df)
num_classes = len(label_encoder.classes_)

train_data, val_data = data.train_val_split(
    train_df,
    val_split=config["val_split"],
    seed=config["seed"],
    stratify_col="ground_truth",
)

train_loader, val_loader = data.create_dataloaders(
    train_data,
    val_data,
    img_dir=config["data_dir"] / "train" / "train",
    input_size=config["input_size"],
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    mean=config["mean"],
    std=config["std"],
)

batch_images, batch_labels = next(iter(train_loader))
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## Setup Model and Training


In [ ]:
model = ArcFaceModel(
    backbone_name=config["backbone_name"],
    num_classes=num_classes,
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    margin=config["arcface_margin"],
    scale=config["arcface_scale"],
    dropout=config["dropout"],
    pretrained=True,
    freeze_backbone=config["freeze_backbone"],
).to(device)

param_count = sum(p.numel() for p in model.parameters())

print("ArcFace Model initialized")
print(f"  Backbone: {config['backbone_name']}")
print(f"  Freeze backbone: {config['freeze_backbone']}")
print(f"  Embedding dim: {config['embedding_dim']}")
print(f"  Hidden dim: {config['hidden_dim']}")
print(f"  Num classes: {num_classes}")
print(f"  Total parameters: {param_count:,}")


## Setup W&B


In [ ]:
run_name = f"{config['backbone_name'].split('/')[-1]}-arcface"
wandb = init_wandb(config, run_name=run_name, param_count=param_count)


In [ ]:
model.eval()
with torch.no_grad():
    logits, embeddings = model(batch_images.to(device), batch_labels.to(device))

print(f"Logits shape: {logits.shape}")
print(f"Embeddings shape: {embeddings.shape}")


In [ ]:
criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
)


In [ ]:
results = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    label_encoder=label_encoder,
    wandb_run=wandb,
    checkpoint_name="arcface_best.pth",
)

history = results["history"]

wandb.run.summary["best_val_mAP"] = results["best_map"]
wandb.run.summary["best_val_loss"] = results["best_val_loss"]
wandb.run.summary["best_epoch"] = results["best_epoch"]
wandb.run.summary["total_epochs"] = results["epochs_ran"]


## Create Kaggle Submission


In [ ]:
pairs_df = data.load_test_pairs_df(config["data_dir"])
unique_images = set(pairs_df["query_image"].unique()) | set(pairs_df["gallery_image"].unique())
unique_images = sorted(list(unique_images))

checkpoint_path = config["checkpoint_dir"] / "arcface_best.pth"
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']}")


In [ ]:
test_df = pd.DataFrame({"filename": unique_images})

test_loader = data.create_test_loader(
    model,
    test_df,
    img_dir=config["data_dir"] / "test" / "test",
    input_size=config["input_size"],
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    mean=config["mean"],
    std=config["std"],
)

names, embeddings = inference.extract_embeddings_with_names(model, test_loader, device)
embedding_lookup = inference.build_embedding_lookup(names, embeddings)

similarities = inference.compute_similarity_for_pairs(pairs_df, embedding_lookup)

submission_df = pd.DataFrame({
    "row_id": pairs_df["row_id"],
    "similarity": similarities,
})

submission_path = config["checkpoint_dir"] / "submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")


In [ ]:
# Save model + submission as W&B artifacts
log_checkpoint_artifact(
    wandb,
    config["checkpoint_dir"] / "arcface_best.pth",
    artifact_name="arcface-model",
    description="ArcFace model checkpoint",
)

log_submission_artifact(
    wandb,
    submission_path,
    artifact_name="submission",
    description="Competition submission file",
)

wandb.finish()
